In [1]:
import pandas as pd

In [4]:
columns = ['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
           'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
           'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
           'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
           'tunnel_parents', 'ip_proto']
df_exfil = pd.read_csv("../data/raw/zeek-captures/exfil_conn.log",
                       sep='\t', comment='#', names=columns)
df_exfil.shape

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
0,1.791071e+09,CQXpGG2GmAxSLgbXkg,192.168.150.129,56981,8.8.8.8,53,udp,dns,0.034184,0,...,T,F,0,Cd,0,0,1,89,-,17
1,1.791071e+09,CfnIIokYXY2HlCKJd,192.168.150.129,35894,8.8.8.8,53,udp,dns,0.035858,0,...,T,F,0,Cd,0,0,1,101,-,17
2,1.791071e+09,CdezjRWhmZW7hCdhf,192.168.150.129,57242,8.8.8.8,53,udp,dns,0.035980,0,...,T,F,0,Cd,0,0,1,422,-,17
3,1.791071e+09,CsGpEo9ikYKe0eCCl,fe80::20c:29ff:fe6c:6d6c,5353,ff02::fb,5353,udp,-,-,-,...,T,F,0,C,0,0,0,0,-,17
4,1.791071e+09,CBRvFK1eyVFzaTakTh,192.168.150.129,5353,224.0.0.251,5353,udp,-,-,-,...,T,T,0,C,0,0,0,0,-,17


In [5]:
df_exfil['orig_ip_bytes'] = pd.to_numeric(df_exfil['orig_ip_bytes'], errors='coerce').fillna(0)
df_exfil['resp_ip_bytes'] = pd.to_numeric(df_exfil['resp_ip_bytes'], errors='coerce').fillna(0)

In [6]:
df_exfil['total_bytes'] = df_exfil['orig_ip_bytes'] + df_exfil['resp_ip_bytes']

In [7]:
volume_by_dest = df_exfil.groupby('id.resp_h')['total_bytes'].sum().sort_values(ascending=False)
print(volume_by_dest)


id.resp_h
18.233.182.23      32508855
239.255.255.250        2779
104.20.23.154          2360
ff02::c                2091
8.8.8.8                1182
172.66.147.243         1180
ff02::2                  48
224.0.0.251               0
ff02::fb                  0
Name: total_bytes, dtype: int64


In [8]:
threshold = 1_000_000   # 1 MB
exfil_detected = volume_by_dest[volume_by_dest > threshold]
print(exfil_detected)

id.resp_h
18.233.182.23    32508855
Name: total_bytes, dtype: int64


Exfiltration detection by total volume per destination. The bulk transfer (~32MB) stands out trivially against normal traffic (max ~2.8KB). Note: the transferred bytes appeared in resp_bytes rather than orig_bytes (the test endpoint echoed the upload, and NAT capture affected byte accounting), so the detector sums total bytes in both directions to remain direction-agnostic. Limitation: volume thresholds are context-dependent — real detection needs a per-destination baseline of normal volume, and sophisticated exfil evades volume detection by trickling data out slowly in small chunks (where timing/beaconing detection becomes relevant).